In [1]:
# Setting up the file path
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

In [2]:
# Importing packages and modules
import comet_ml
from openpyxl import Workbook, load_workbook
from datetime import datetime
import torch
from matplotlib import pyplot as plt
from pytorch_lightning.loggers import CometLogger
from copy import deepcopy
import numpy as np
import gymnasium as gym
from itertools import product
from tqdm import tqdm
from RL4CRN_Feedback.Input_Output_Rxn_Networks.IOCRN_MassAction import IOCRN_MassAction
from RL4CRN_Feedback.Environments.CRNEnvironment import CRNEnvironment
from RL4CRN_Feedback.Environments.VecCRNEnvironment import VecCRNEnvironment
from RL4CRN_Feedback.Environments.VecCRNEnvironment import SerialVecCRNEnvironment
from RL4CRN_Feedback.Agents.PPOAgent import PPOAgent
from RL4CRN_Feedback.Policies.BimolecularMassActionPolicy import BimolecularMassActionPolicy
from RL4CRN_Feedback.Policies.BimolecularMassActionValue import BimolecularMassActionValue
from RL4CRN_Feedback.Policies.PPO import PPO
from RL4CRN_Feedback.Rewards.Transients import dynamic_tracking_error

In [3]:
# Set the logger to use Comet
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
api_key = "o77J6VCMDamustkfJuMXZ2jdV" #"vhIR3uyqsKyU4L7SA8fLCfTSC" 
logger = CometLogger(
    api_key=api_key,
    project="Molecular_Integrators",        
    workspace= "maurice-filo", #"redsnic"
    name=f'Transients_Experiment_{timestamp}',
)
logger = logger.experiment

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/maurice-filo/molecular-integrators/9715547ea25f4df4b2cc37e2625cb5fd



In [4]:
# Construct the template CRN
species_labels = ['X_1', 'X_2', 'X_3', 'X_4']
inputs_labels = ['u_1', 'u_2', 'u_3']
stoichiometry_reactants = np.array([[0, 0, 0, 1, 0, 0, 0], [0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 1]], dtype=np.int8)
stoichiometry_products = np.array([[1, 0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]], dtype=np.int8)
parameters = np.array([1, 1, 1, 1, 1, 1, 1], dtype=np.float32)
input_influence_matrix = np.array([[1, 0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0],  [0, 0, 1, 0, 0, 0, 0]], dtype=np.int8)
outputs = np.array([4], dtype=np.int8)
CRN_template = IOCRN_MassAction(stoichiometry_reactants, stoichiometry_products, parameters, input_influence_matrix, outputs, species_labels, inputs_labels)
print('CRN template:')
CRN_template.print_reactions()
num_species = len(species_labels)
num_inputs = len(inputs_labels)

CRN template:
Inputs: ['u_1', 'u_2', 'u_3'] 
Species: ['X_1', 'X_2', 'X_3', 'X_4'] 
Output Species: ['X_4'] 
Reaction 0: 0 -> X_1 ; Rate Constant: 1.0u_1 
Reaction 1: 0 -> X_2 ; Rate Constant: 1.0u_2 
Reaction 2: 0 -> X_3 ; Rate Constant: 1.0u_3 
Reaction 3: X_1 -> 0 ; Rate Constant: 1.0 
Reaction 4: X_2 -> 0 ; Rate Constant: 1.0 
Reaction 5: X_3 -> 0 ; Rate Constant: 1.0 
Reaction 6: X_4 -> 0 ; Rate Constant: 1.0 



In [5]:
# Hyperparameters
max_num_reactions = 5                               # Maximum number of reactions
N_CPUs = 128                                        # Number of CPUs          
n_samples = 20*N_CPUs                               # Number of samples    
width = 1024
depth = 5
hidden_size = 1024
allow_input_influence = False
learning_rate = 1e-4
entropy_weight = 0.0001
entropy_update_coefficient = 0.75
entropy_schedule = 10
minimum_entropy_weight = 0.0001
risk = 0.99
risk_update = 0
max_risk = 1.0
risk_schedule = 20
epoch_num = 10000
render_schedule = 5
t_f = 100
mode = {'style': 'logger', 'task': 'transients', 'format': 'image'}

# Define logic function
def logic_function(x):
    return (x[:,0] & x[:,1]) | (~x[:,1] & x[:,2])

# Create the reward function
def compute_reward(state):
    nums = [0, 1]
    u = np.array(list(product(nums, repeat=state.num_inputs)), dtype=np.int32)
    initial_condition = np.array([0, 0, 0, 0], dtype=np.float32)
    time_horizon = np.linspace(0, t_f, 1000, dtype=np.float32)
    r = logic_function(u).astype(np.float32)
    return dynamic_tracking_error(state, u, initial_condition, time_horizon, r, threshold=1000, norm=1)

# Sheet File
file_name = "Logic_Gate1.xlsx"

In [6]:
sheet_name = "Data"
headers = ["Timestamp", "URL", "Maximum Number of Reactions", "Number of Species", "Number of Inputs", "Number of Samples", "Final Time", 
           "Allow Input Influence",
           "Learning Rate", "Initial Entropy Weight", "Entropy Update Coefficient", "Entropy Schedule",
           "Minimum Entropy Weight", "Risk", "Risk Update", "Maximum Risk", "Risk Schedule",
           "Number of Epochs", "Render Schedule", "Neural Network Depth", "Neural Network Width", "Number of CPUs"]
data_row = [timestamp, logger.url, max_num_reactions, num_species, num_inputs, n_samples, t_f,
            allow_input_influence, 
            learning_rate, entropy_weight, entropy_update_coefficient, entropy_schedule, 
            minimum_entropy_weight, risk, risk_update, max_risk, risk_schedule,
            epoch_num, render_schedule, depth, width, N_CPUs]

if os.path.exists(file_name):
    wb = load_workbook(file_name)
    if sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
    else:
        ws = wb.create_sheet(sheet_name)
else:
    wb = Workbook()
    ws = wb.active
    ws.title = sheet_name

if ws.max_column < 2:
    for i, header in enumerate(headers, start=1):
        ws.cell(row=i, column=1, value=header)
next_col = ws.max_column + 1
for i, value in enumerate(data_row, start=1):
    ws.cell(row=i, column=next_col, value=value)

wb.save(file_name)
print(f"New experiment data saved in column {next_col} of '{file_name}'.")

New experiment data saved in column 6 of 'Logic_Gate1.xlsx'.


In [7]:
# Construct parallel environments
CRN_0 = deepcopy(CRN_template)
vec_env = VecCRNEnvironment([CRNEnvironment(CRN_0, max_num_reactions, logger=logger, logger_schedule=1) for _ in range(n_samples)], N_CPUs=N_CPUs, logger=logger)
# vec_env = SerialVecCRNEnvironment([CRNEnvironment(CRN_0, max_num_reactions, logger=logger, logger_schedule=1) for _ in range(n_samples)], logger=logger)

In [8]:
# Construct the Agent
device = 'cuda' if torch.cuda.is_available() else 'cpu'
num_species = len(species_labels); num_inputs = len(inputs_labels)
encoder_attributes = {"hidden_size": width, "num_layers": depth}
structure_decoder_attributes = {"hidden_size": width, "num_layers": depth}
rate_decoder_attributes = {"hidden_size": width, "num_layers": depth}
input_influence_decoder_attributes = {"hidden_size": width, "num_layers": depth}
num_possible_reactions = CRN_0.get_reactions_range()
policy = BimolecularMassActionPolicy(num_possible_reactions, num_inputs, encoder_attributes, hidden_size, structure_decoder_attributes, rate_decoder_attributes, input_influence_decoder_attributes, continuous_distribution='lognormal', allow_input_influence=False, device=device)
v_net = BimolecularMassActionValue(num_possible_reactions, num_inputs, encoder_attributes, hidden_size, structure_decoder_attributes, rate_decoder_attributes, input_influence_decoder_attributes, continuous_distribution='lognormal', allow_input_influence=False, device=device)
ppo_policy = PPO(policy, v_net, eps=0.2, device=device, entropy_weight=0.01, logger=logger)
agent = PPOAgent(vec_env.envs[0], ppo_policy, allow_input_influence=False, logger=logger, learning_rate=learning_rate, entropy_weight=entropy_weight, entropy_update_coefficient=entropy_update_coefficient, entropy_schedule=entropy_schedule, minimum_entropy_weight=minimum_entropy_weight, risk=risk, risk_update=risk_update, max_risk=max_risk, risk_schedule=risk_schedule, device=device,
                accumulate_gradients_for=1, update_policy_every=5)

In [ ]:
# Training Loop

import numpy as np

for i in tqdm(range(epoch_num)):
    vec_env.reset()
    observations = []
    actions = [] 
    rewards = []
    observations = [vec_env.observe()]
    log_probs = []
    entropies = []

    loop_limit = max_num_reactions + vec_env.envs[0].CRN_template.num_unknown_parameters
    for j in range(loop_limit):
        observations.append(vec_env.observe())
        action, log_prob, entropy = agent.act(observations[-1])
        actions.append(action)
        log_probs.append(log_prob)
        entropies.append(entropy)
        vec_env.step(action, mode='reaction index')
        if j < loop_limit - 1:
            rewards.append([0.0] * len(vec_env.envs))
        else:
            rewards.append(vec_env.get_reward(compute_reward))
            logger.log_metric("Raw Reward MEAN", sum(rewards[-1])/len(rewards[-1]), step=i)
            logger.log_metric("Raw Reward MAX", max(rewards[-1]), step=i)
            logger.log_metric("Raw Reward MIN", min(rewards[-1]), step=i)
    
    
    rewards_torch = torch.tensor(rewards, dtype=torch.float32, device=device)

    entropies =  torch.stack(entropies, dim=0).to(device)
    losses = agent.policy.reward(observations, actions, rewards_torch, entropies, gamma=1., lam=0.95, value_loss_weight=0.5)

    agent.update(losses, scores=rewards_torch.mean(dim=0))
    
    if i % render_schedule == 0:
        vec_env.render(rewards[-1], mode=mode)

 14%|█▍        | 1405/10000 [2:16:17<13:59:09,  5.86s/it]

In [ ]:
# plot best CRN

plt.plot(rewards[-2])

In [ ]:
# get the best CRN
import random
best_index = 1643 #  random.randint(0, len(vec_env.envs))
print(best_index)
best_crn = deepcopy(vec_env.envs[best_index].state)
best_crn.print_reactions()

# make all combinations of u of 0.5, 1., 1.5 (like 0.5,0.5 : 0.5,1.0 and so on)
# u = np.array(list(product([0.5, 1., 1.5], repeat=best_crn.num_inputs)), dtype=np.float32)

u = np.array([[1., i*0.1+0.5] for i in range(20)])

initial_condition = np.array([0, 0, 0], dtype=np.float32)
time_horizon = np.linspace(0, 200, 1000, dtype=np.float32)

best_crn.transient_response(inputs=u, initial_condition=initial_condition, time_horizon=time_horizon, return_states=True)
best_crn.plot_transient_response()